<a href="https://colab.research.google.com/github/yashshinde0080/RLM_yashshinde0080-zeronet/blob/main/zeronet_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# CELL 1: Install dependencies
# ============================================================
# Run this once, then restart runtime if prompted

!pip install -q torch torchvision torchaudio
!pip install -q tokenizers datasets huggingface_hub transformers
!pip install -q bitsandbytes accelerate sentencepiece
!pip install -q wandb  # optional, for logging

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.9.0+cu126
CUDA available: True
GPU: Tesla T4
VRAM: 15.8 GB


In [3]:
# ============================================================
# CELL 2: All hyperparameters in one place
# ============================================================
# Change things HERE, not scattered across 30 cells

from dataclasses import dataclass

@dataclass
class Config:
    # Model architecture
    vocab_size: int = 32000
    d_model: int = 512
    n_heads: int = 8
    n_layers: int = 6
    max_len: int = 256
    mlp_ratio: int = 4
    dropout: float = 0.1

    # Training
    batch_size: int = 8
    learning_rate: float = 3e-4
    weight_decay: float = 0.1
    epochs: int = 5
    warmup_steps: int = 500
    max_grad_norm: float = 1.0
    fp16: bool = True

    # Data
    dataset_name: str = "roneneldan/TinyStories"
    max_train_samples: int = 500_000    # limit for Colab
    max_val_samples: int = 5_000

    # Paths
    tokenizer_dir: str = "zeronet1-tokenizer"
    model_dir: str = "zeronet1-model"
    chat_model_dir: str = "zeronet1-chat"

    # Hugging Face
    hf_repo: str = "YOUR_USERNAME/zeronet-1"  # CHANGE THIS

    @property
    def head_dim(self):
        return self.d_model // self.n_heads

config = Config()

# Sanity checks
assert config.d_model % config.n_heads == 0, "d_model must be divisible by n_heads"
assert config.head_dim % 2 == 0, "head_dim must be even for RoPE"

print("Configuration loaded.")
print(f"  d_model={config.d_model}, heads={config.n_heads}, layers={config.n_layers}")
print(f"  head_dim={config.head_dim}, max_len={config.max_len}")
print(f"  Estimated params: ~{(config.n_layers * (4 * config.d_model**2 + 2 * config.d_model * config.d_model * config.mlp_ratio) + config.vocab_size * config.d_model) / 1e6:.0f}M")

Configuration loaded.
  d_model=512, heads=8, layers=6
  head_dim=64, max_len=256
  Estimated params: ~35M


In [4]:
# ============================================================
# CELL 3: Load and inspect dataset
# ============================================================
# Dataset stays on HF servers, streams/caches automatically
# No manual downloads, no unzipping, no file management

from datasets import load_dataset

print("Loading TinyStories dataset from Hugging Face...")
raw_dataset = load_dataset(
    config.dataset_name,
    split="train",
    streaming=False  # set True if RAM is tight
)

# Take a subset for Colab
print(f"Full dataset size: {len(raw_dataset)}")
train_raw = raw_dataset.select(range(min(config.max_train_samples, len(raw_dataset))))
print(f"Using {len(train_raw)} samples for training")

# Validation split
val_dataset_full = load_dataset(config.dataset_name, split="validation")
val_raw = val_dataset_full.select(range(min(config.max_val_samples, len(val_dataset_full))))
print(f"Using {len(val_raw)} samples for validation")

# Inspect
print("\n--- Sample ---")
print(train_raw[0]["text"][:500])

Loading TinyStories dataset from Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

...

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Full dataset size: 2119719
Using 500000 samples for training


Using 5000 samples for validation

--- Sample ---
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them b


In [5]:
# ============================================================
# CELL 4: Train BPE tokenizer from scratch
# ============================================================
# No pretrained tokenizer. You build this yourself.

import os
from tokenizers import (
    Tokenizer,
    models,
    trainers,
    pre_tokenizers,
    decoders,
    processors,
)

def train_tokenizer(dataset, config):
    """Train a BPE tokenizer on the dataset text."""

    # Write text to temp file (tokenizers library needs files)
    temp_file = "temp_tokenizer_data.txt"
    print("Writing text to temp file...")
    with open(temp_file, "w", encoding="utf-8") as f:
        for i, example in enumerate(dataset):
            text = example["text"].strip()
            if text:
                f.write(text + "\n")
            if i % 100_000 == 0 and i > 0:
                print(f"  Written {i} samples...")

    print(f"Temp file ready. Training tokenizer with vocab_size={config.vocab_size}...")

    # Build tokenizer
    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()

    # Special tokens
    special_tokens = [
        "<pad>",
        "<unk>",
        "<bos>",
        "<eos>",
        "<|user|>",
        "<|assistant|>",
    ]

    trainer = trainers.BpeTrainer(
        vocab_size=config.vocab_size,
        min_frequency=2,
        special_tokens=special_tokens,
        show_progress=True,
    )

    tokenizer.train([temp_file], trainer)

    # Post-processor: add BOS/EOS
    bos_id = tokenizer.token_to_id("<bos>")
    eos_id = tokenizer.token_to_id("<eos>")
    tokenizer.post_processor = processors.TemplateProcessing(
        single=f"<bos>:0 $A:0 <eos>:0",
        pair=f"<bos>:0 $A:0 <eos>:0 <bos>:1 $B:1 <eos>:1",
        special_tokens=[
            ("<bos>", bos_id),
            ("<eos>", eos_id),
        ],
    )

    # Enable padding
    tokenizer.enable_padding(
        pad_id=tokenizer.token_to_id("<pad>"),
        pad_token="<pad>",
        length=config.max_len,
    )

    # Enable truncation
    tokenizer.enable_truncation(max_length=config.max_len)

    # Save
    os.makedirs(config.tokenizer_dir, exist_ok=True)
    tokenizer.save(os.path.join(config.tokenizer_dir, "tokenizer.json"))

    # Cleanup
    os.remove(temp_file)

    print(f"Tokenizer trained and saved to {config.tokenizer_dir}/")
    print(f"Vocab size: {tokenizer.get_vocab_size()}")

    return tokenizer

# Train it
tokenizer = train_tokenizer(train_raw, config)

# Update config with actual vocab size
config.vocab_size = tokenizer.get_vocab_size()

# Test
test_encoding = tokenizer.encode("The cat sat on the mat.")
print(f"\nTest: 'The cat sat on the mat.'")
print(f"Tokens: {test_encoding.tokens}")
print(f"IDs: {test_encoding.ids}")

# Verify special tokens
for tok in ["<pad>", "<unk>", "<bos>", "<eos>", "<|user|>", "<|assistant|>"]:
    tid = tokenizer.token_to_id(tok)
    print(f"  {tok} -> {tid}")

Writing text to temp file...
  Written 100000 samples...
  Written 200000 samples...
  Written 300000 samples...
  Written 400000 samples...
Temp file ready. Training tokenizer with vocab_size=32000...
Tokenizer trained and saved to zeronet1-tokenizer/
Vocab size: 32000

Test: 'The cat sat on the mat.'
Tokens: ['<bos>', 'The', 'Ġcat', 'Ġsat', 'Ġon', 'Ġthe', 'Ġmat', '.', '<eos>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>

In [6]:
# ============================================================
# CELL 5: Tokenize dataset and create DataLoaders
# ============================================================

from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    """
    Tokenizes text and creates input/target pairs
    for next-token prediction.
    """

    def __init__(self, hf_dataset, tokenizer, max_len):
        self.max_len = max_len
        self.examples = []

        print("Tokenizing dataset...")
        for i, example in enumerate(hf_dataset):
            text = example["text"].strip()
            if not text:
                continue

            encoded = tokenizer.encode(text)
            ids = encoded.ids

            # Only keep sequences that have enough tokens
            if len(ids) >= 4:
                # Pad or truncate to max_len + 1
                # (+1 because we need input AND target)
                ids = ids[: self.max_len + 1]
                if len(ids) < self.max_len + 1:
                    pad_id = tokenizer.token_to_id("<pad>")
                    ids = ids + [pad_id] * (self.max_len + 1 - len(ids))

                self.examples.append(ids)

            if i % 100_000 == 0 and i > 0:
                print(f"  Processed {i} samples...")

        print(f"Dataset ready: {len(self.examples)} examples")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ids = self.examples[idx]
        # Input: all tokens except last
        # Target: all tokens except first (shifted by 1)
        x = torch.tensor(ids[:-1], dtype=torch.long)
        y = torch.tensor(ids[1:], dtype=torch.long)
        return x, y


# Build datasets
train_dataset = TextDataset(train_raw, tokenizer, config.max_len)
val_dataset = TextDataset(val_raw, tokenizer, config.max_len)

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)

# Verify shapes
x_sample, y_sample = next(iter(train_loader))
print(f"\nBatch shapes:")
print(f"  Input:  {x_sample.shape}")   # [batch, max_len]
print(f"  Target: {y_sample.shape}")   # [batch, max_len]
print(f"\nTotal batches per epoch: {len(train_loader)}")

Tokenizing dataset...
  Processed 100000 samples...
  Processed 200000 samples...
  Processed 300000 samples...
  Processed 400000 samples...
Dataset ready: 499945 examples
Tokenizing dataset...
Dataset ready: 5000 examples

Batch shapes:
  Input:  torch.Size([8, 256])
  Target: torch.Size([8, 256])

Total batches per epoch: 62493


In [7]:
# ============================================================
# CELL 6: Rotary Positional Embeddings (RoPE)
# ============================================================
# This replaces absolute positional embeddings entirely.
# Applied to Q and K only. Never V.

import torch
import torch.nn as nn
import math

def precompute_rope_freqs(head_dim: int, max_len: int, base: float = 10000.0):
    """
    Precompute the rotation frequencies for RoPE.

    For each pair of dimensions (2i, 2i+1), compute:
        theta_i = 1 / (base^(2i/d))
    Then for each position pos:
        angle = pos * theta_i

    Returns: (max_len, head_dim/2) tensor of angles
    """
    assert head_dim % 2 == 0, "head_dim must be even"

    # Inverse frequencies: shape (head_dim/2,)
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))

    # Position indices: shape (max_len,)
    positions = torch.arange(max_len).float()

    # Outer product: shape (max_len, head_dim/2)
    freqs = torch.einsum("i,j->ij", positions, inv_freq)

    return freqs


def apply_rope(x: torch.Tensor, freqs: torch.Tensor) -> torch.Tensor:
    """
    Apply rotary embeddings to input tensor.

    x: (batch, n_heads, seq_len, head_dim)
    freqs: (seq_len, head_dim/2)

    For each pair (x1, x2) at dimensions (2i, 2i+1):
        x1_new = x1 * cos(theta) - x2 * sin(theta)
        x2_new = x1 * sin(theta) + x2 * cos(theta)
    """
    # Split into even and odd dimensions
    x1 = x[..., 0::2]  # (batch, heads, seq, head_dim/2)
    x2 = x[..., 1::2]  # (batch, heads, seq, head_dim/2)

    # Reshape freqs to broadcast: (1, 1, seq_len, head_dim/2)
    cos_f = freqs.cos().unsqueeze(0).unsqueeze(0).to(x.device, dtype=x.dtype)
    sin_f = freqs.sin().unsqueeze(0).unsqueeze(0).to(x.device, dtype=x.dtype)

    # Apply rotation
    x1_rot = x1 * cos_f - x2 * sin_f
    x2_rot = x1 * sin_f + x2 * cos_f

    # Interleave back: stack on last dim then flatten
    # Shape: (batch, heads, seq, head_dim/2, 2) -> (batch, heads, seq, head_dim)
    out = torch.stack([x1_rot, x2_rot], dim=-1).flatten(-2)

    return out


# Quick test
print("Testing RoPE...")
test_freqs = precompute_rope_freqs(64, 256)
print(f"  Freq table shape: {test_freqs.shape}")  # (256, 32)

test_tensor = torch.randn(2, 8, 128, 64)
rotated = apply_rope(test_tensor, test_freqs[:128])
print(f"  Input shape:  {test_tensor.shape}")
print(f"  Output shape: {rotated.shape}")
assert test_tensor.shape == rotated.shape, "Shape mismatch!"
print("  RoPE test passed.")

Testing RoPE...
  Freq table shape: torch.Size([256, 32])
  Input shape:  torch.Size([2, 8, 128, 64])
  Output shape: torch.Size([2, 8, 128, 64])
  RoPE test passed.


In [8]:
# ============================================================
# CELL 7: Multi-Head Self-Attention with RoPE + Causal Mask
# ============================================================

class CausalSelfAttention(nn.Module):
    """
    Multi-head self-attention with:
    - RoPE on Q and K (no absolute positional embeddings)
    - Causal mask (no future token leakage)
    - Dropout on attention weights
    """

    def __init__(self, d_model: int, n_heads: int, max_len: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0

        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.d_model = d_model

        # Single projection for Q, K, V (more efficient than 3 separate)
        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        # Precompute RoPE frequencies
        freqs = precompute_rope_freqs(self.head_dim, max_len)
        self.register_buffer("freqs", freqs, persistent=False)

        # Precompute causal mask
        # Lower triangular = 1 means "allowed to attend"
        mask = torch.tril(torch.ones(max_len, max_len))
        self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (batch, seq_len, d_model)
        returns: (batch, seq_len, d_model)
        """
        B, T, C = x.shape

        # Project to Q, K, V
        qkv = self.qkv_proj(x)                       # (B, T, 3*d_model)
        q, k, v = qkv.chunk(3, dim=-1)               # each (B, T, d_model)

        # Reshape to (B, n_heads, T, head_dim)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # Apply RoPE to Q and K only
        rope_freqs = self.freqs[:T]
        q = apply_rope(q, rope_freqs)
        k = apply_rope(k, rope_freqs)

        # Attention scores
        scale = self.head_dim ** 0.5
        scores = (q @ k.transpose(-2, -1)) / scale   # (B, n_heads, T, T)

        # Apply causal mask
        mask = self.causal_mask[:T, :T]               # (T, T)
        scores = scores.masked_fill(mask == 0, float("-inf"))

        # Softmax + dropout
        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)

        # Weighted sum of values
        out = attn_weights @ v                         # (B, n_heads, T, head_dim)

        # Reshape back to (B, T, d_model)
        out = out.transpose(1, 2).contiguous().view(B, T, C)

        return self.resid_dropout(self.out_proj(out))


# Test
print("Testing CausalSelfAttention...")
attn = CausalSelfAttention(512, 8, 256)
test_x = torch.randn(2, 128, 512)
test_out = attn(test_x)
print(f"  Input:  {test_x.shape}")
print(f"  Output: {test_out.shape}")
assert test_x.shape == test_out.shape
print("  Attention test passed.")

Testing CausalSelfAttention...
  Input:  torch.Size([2, 128, 512])
  Output: torch.Size([2, 128, 512])
  Attention test passed.


In [ ]:
# ============================================================
# CELL 8: Feed-Forward Network (SwiGLU variant)
# ============================================================
# SwiGLU is what LLaMA uses. Better than plain ReLU/GELU MLP.
# If you want simpler, swap to the GELU version at the bottom.

class FeedForward(nn.Module):
    """
    SwiGLU feed-forward network.
    Better gradient flow than standard MLP.
    """

    def __init__(self, d_model: int, mlp_ratio: int = 4, dropout: float = 0.1):
        super().__init__()
        hidden = d_model * mlp_ratio

        self.w1 = nn.Linear(d_model, hidden, bias=False)
        self.w2 = nn.Linear(hidden, d_model, bias=False)
        self.w3 = nn.Linear(d_model, hidden, bias=False)  # gate
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # SwiGLU: (swish(W1·x)) * (W3·x)
        swish = torch.nn.functional.silu(self.w1(x))
        gate = self.w3(x)
        return self.dropout(self.w2(swish * gate))


# Alternative: standard GELU MLP (uncomment if SwiGLU causes issues)
# class FeedForward(nn.Module):
#     def __init__(self, d_model, mlp_ratio=4, dropout=0.1):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(d_model, d_model * mlp_ratio, bias=False),
#             nn.GELU(),
#             nn.Linear(d_model * mlp_ratio, d_model, bias=False),
#             nn.Dropout(dropout),
#         )
#     def forward(self, x):
#         return self.net(x)


print("FeedForward module ready.")
ff_test = FeedForward(512)
ff_out = ff_test(torch.randn(2, 128, 512))
print(f"  FFN output shape: {ff_out.shape}")

In [ ]:
# ============================================================
# CELL 9: Single Transformer Block
# ============================================================
# Pre-norm architecture (LayerNorm before attention/FFN)
# This is what GPT-2+, LLaMA, etc. use.
# Post-norm (original Vaswani) is harder to train.

class TransformerBlock(nn.Module):
    """
    Pre-norm Transformer block:
        x -> LN -> Attention -> Residual
        x -> LN -> FFN -> Residual
    """

    def __init__(self, d_model: int, n_heads: int, max_len: int,
                 mlp_ratio: int = 4, dropout: float = 0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, max_len, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, mlp_ratio, dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pre-norm + residual
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


print("TransformerBlock ready.")
block = TransformerBlock(512, 8, 256)
block_out = block(torch.randn(2, 128, 512))
print(f"  Block output shape: {block_out.shape}")

In [ ]:
# ============================================================
# CELL 10: Complete Decoder-Only Transformer (ZeroNet-1)
# ============================================================
# No positional embedding layer. RoPE handles position.

class ZeroNet1(nn.Module):
    """
    Decoder-only causal Transformer language model.

    Architecture:
        Token Embedding (no position embedding — RoPE inside attention)
        → N × TransformerBlock
        → LayerNorm
        → Linear head → logits over vocabulary
    """

    def __init__(self, config: Config):
        super().__init__()
        self.config = config

        # Token embedding only (RoPE replaces positional embedding)
        self.token_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.emb_dropout = nn.Dropout(config.dropout)

        # Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=config.d_model,
                n_heads=config.n_heads,
                max_len=config.max_len,
                mlp_ratio=config.mlp_ratio,
                dropout=config.dropout,
            )
            for _ in range(config.n_layers)
        ])

        # Final layer norm
        self.ln_f = nn.LayerNorm(config.d_model)

        # Output head (tied with embedding weights)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

        # Weight tying: embedding and output share weights
        self.lm_head.weight = self.token_emb.weight

        # Initialize weights
        self.apply(self._init_weights)

        # Count parameters
        n_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"ZeroNet-1 initialized: {n_params / 1e6:.2f}M parameters")

    def _init_weights(self, module):
        """Xavier-style initialization."""
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, input_ids: torch.Tensor, targets: torch.Tensor = None):
        """
        input_ids: (batch, seq_len)
        targets: (batch, seq_len) or None

        Returns:
            logits: (batch, seq_len, vocab_size)
            loss: scalar if targets provided, else None
        """
        B, T = input_ids.shape

        # Token embeddings (no positional embedding added)
        x = self.token_emb(input_ids)     # (B, T, d_model)
        x = self.emb_dropout(x)

        # Pass through all transformer blocks
        for block in self.blocks:
            x = block(x)

        # Final norm + project to vocab
        x = self.ln_f(x)
        logits = self.lm_head(x)          # (B, T, vocab_size)

        # Compute loss if targets provided
        loss = None
        if targets is not None:
            # Flatten for cross-entropy
            loss = torch.nn.functional.cross_entropy(
                logits.view(-1, config.vocab_size),
                targets.view(-1),
                ignore_index=tokenizer.token_to_id("<pad>"),
            )

        return logits, loss


# Build the model
model = ZeroNet1(config)

# Move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Model on: {device}")

# Quick forward pass test
test_ids = torch.randint(0, config.vocab_size, (2, config.max_len)).to(device)
test_targets = torch.randint(0, config.vocab_size, (2, config.max_len)).to(device)
test_logits, test_loss = model(test_ids, test_targets)
print(f"  Logits shape: {test_logits.shape}")
print(f"  Loss: {test_loss.item():.4f}")
print("  Forward pass test passed.")

In [ ]:
# ============================================================
# CELL 11: Cosine LR scheduler with warmup
# ============================================================

def get_lr(step, total_steps, warmup_steps, max_lr, min_lr=1e-5):
    """
    Cosine annealing with linear warmup.
    This is the standard schedule for LLM training.
    """
    # Warmup phase: linear increase
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps

    # Cosine decay phase
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))


# Visualize the schedule
total_steps = len(train_loader) * config.epochs
lrs = [get_lr(s, total_steps, config.warmup_steps, config.learning_rate) for s in range(total_steps)]

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 3))
plt.plot(lrs)
plt.xlabel("Step")
plt.ylabel("Learning Rate")
plt.title("LR Schedule: Cosine with Warmup")
plt.grid(True, alpha=0.3)
plt.show()

print(f"Total training steps: {total_steps}")
print(f"Warmup steps: {config.warmup_steps}")

In [ ]:
# ============================================================
# CELL 12: Training loop (the real work happens here)
# ============================================================

import time
from torch.cuda.amp import GradScaler, autocast

def train(model, train_loader, val_loader, config, tokenizer):
    """Full training loop with mixed precision, gradient clipping, and validation."""

    model.train()

    # Optimizer: AdamW with weight decay
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.learning_rate,
        betas=(0.9, 0.95),
        weight_decay=config.weight_decay,
    )

    # Mixed precision scaler
    scaler = GradScaler(enabled=config.fp16)

    total_steps = len(train_loader) * config.epochs
    global_step = 0
    best_val_loss = float("inf")
    train_losses = []
    val_losses = []

    pad_id = tokenizer.token_to_id("<pad>")

    print(f"\n{'='*60}")
    print(f"TRAINING START")
    print(f"  Epochs: {config.epochs}")
    print(f"  Batch size: {config.batch_size}")
    print(f"  Total steps: {total_steps}")
    print(f"  FP16: {config.fp16}")
    print(f"{'='*60}\n")

    for epoch in range(config.epochs):
        epoch_start = time.time()
        epoch_loss = 0.0
        num_batches = 0

        model.train()

        for batch_idx, (input_ids, targets) in enumerate(train_loader):
            input_ids = input_ids.to(device)
            targets = targets.to(device)

            # Update learning rate
            lr = get_lr(global_step, total_steps, config.warmup_steps, config.learning_rate)
            for param_group in optimizer.param_groups:
                param_group["lr"] = lr

            # Forward pass with mixed precision
            with autocast(device_type="cuda", enabled=config.fp16):
                logits, loss = model(input_ids, targets)

            # Backward pass
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()

            # Gradient clipping (prevents explosion)
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(), config.max_grad_norm
            )

            scaler.step(optimizer)
            scaler.update()

            # Track
            epoch_loss += loss.item()
            num_batches += 1
            global_step += 1

            # Log every 100 steps
            if global_step % 100 == 0:
                avg_loss = epoch_loss / num_batches
                print(
                    f"  Step {global_step:>6d}/{total_steps} | "
                    f"Loss: {loss.item():.4f} | "
                    f"Avg: {avg_loss:.4f} | "
                    f"LR: {lr:.2e} | "
                    f"Grad: {grad_norm:.2f}"
                )

        # Epoch stats
        avg_train_loss = epoch_loss / num_batches
        train_losses.append(avg_train_loss)
        epoch_time = time.time() - epoch_start

        # Validation
        val_loss = evaluate(model, val_loader, config)
        val_losses.append(val_loss)

        print(f"\n  Epoch {epoch+1}/{config.epochs} complete in {epoch_time:.0f}s")
        print(f"  Train loss: {avg_train_loss:.4f}")
        print(f"  Val loss:   {val_loss:.4f}")
        print(f"  Perplexity: {math.exp(val_loss):.2f}")

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(model, optimizer, epoch, global_step, config)
            print(f"  ✓ Best model saved (val_loss={val_loss:.4f})")

        # Generate sample text
        print(f"\n  --- Sample Generation ---")
        sample = generate_text(model, tokenizer, "Once upon a time", max_len=80)
        print(f"  {sample}")
        print(f"  {'─'*50}\n")

    # Plot training curves
    plt.figure(figsize=(10, 4))
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("ZeroNet-1 Training Curves")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    return train_losses, val_losses


@torch.no_grad()
def evaluate(model, val_loader, config):
    """Compute average validation loss."""
    model.eval()
    total_loss = 0.0
    num_batches = 0

    for input_ids, targets in val_loader:
        input_ids = input_ids.to(device)
        targets = targets.to(device)

        with autocast(device_type="cuda", enabled=config.fp16):
            _, loss = model(input_ids, targets)

        total_loss += loss.item()
        num_batches += 1

    model.train()
    return total_loss / max(num_batches, 1)


def save_checkpoint(model, optimizer, epoch, step, config):
    """Save model checkpoint."""
    import os
    os.makedirs(config.model_dir, exist_ok=True)

    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": epoch,
        "step": step,
        "config": config,
    }, os.path.join(config.model_dir, "checkpoint.pt"))


print("Training functions defined. Ready to train.")

In [ ]:
# ============================================================
# CELL 13: Text generation with sampling strategies
# ============================================================

@torch.no_grad()
def generate_text(
    model,
    tokenizer,
    prompt: str,
    max_len: int = 200,
    temperature: float = 0.8,
    top_k: int = 50,
    top_p: float = 0.9,
):
    """
    Generate text using top-k + top-p (nucleus) sampling.
    No beam search — this is a language model, not a crossword solver.
    """
    model.eval()

    # Encode prompt
    encoded = tokenizer.encode(prompt)
    input_ids = encoded.ids

    # Remove trailing pad/eos from tokenizer post-processing
    pad_id = tokenizer.token_to_id("<pad>")
    eos_id = tokenizer.token_to_id("<eos>")

    # Keep only meaningful tokens
    input_ids = [t for t in input_ids if t != pad_id]

    generated = list(input_ids)

    for _ in range(max_len):
        # Take last max_len tokens as context
        context = generated[-config.max_len:]
        x = torch.tensor([context], dtype=torch.long).to(device)

        # Forward pass
        logits, _ = model(x)
        logits = logits[0, -1, :]  # Last token's logits

        # Temperature scaling
        logits = logits / temperature

        # Top-k filtering
        if top_k > 0:
            top_k_vals, _ = torch.topk(logits, top_k)
            threshold = top_k_vals[-1]
            logits[logits < threshold] = float("-inf")

        # Top-p (nucleus) filtering
        if top_p < 1.0:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.softmax(sorted_logits, dim=-1).cumsum(dim=-1)

            # Remove tokens with cumulative prob above threshold
            sorted_mask = cumulative_probs - torch.softmax(sorted_logits, dim=-1) >= top_p
            sorted_logits[sorted_mask] = float("-inf")

            # Scatter back
            logits = torch.zeros_like(logits).scatter(0, sorted_indices, sorted_logits)

        # Sample
        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1).item()

        # Stop on EOS
        if next_token == eos_id:
            break

        generated.append(next_token)

    # Decode
    text = tokenizer.decode(generated)
    model.train()
    return text


# We'll test this after training
print("Generation function ready.")

In [ ]:
# ============================================================
# CELL 14: TRAIN THE MODEL
# ============================================================
# This is where you wait. Get coffee.
# On Colab T4: ~15-40 min per epoch depending on data size.

train_losses, val_losses = train(model, train_loader, val_loader, config, tokenizer)

In [ ]:
# ============================================================
# CELL 15: Generate text with trained model
# ============================================================

prompts = [
    "Once upon a time",
    "The little dog",
    "She looked at the sky and",
    "There was a boy who loved",
    "The forest was dark and",
]

print("=" * 60)
print("ZERONET-1: TEXT GENERATION")
print("=" * 60)

for prompt in prompts:
    print(f"\nPrompt: '{prompt}'")
    print("-" * 40)

    # Temperature 0.7 (balanced)
    text = generate_text(model, tokenizer, prompt, max_len=150, temperature=0.7)
    print(f"[T=0.7] {text}")

    # Temperature 1.0 (more creative)
    text = generate_text(model, tokenizer, prompt, max_len=150, temperature=1.0)
    print(f"[T=1.0] {text}")

    print()

In [ ]:
# ============================================================
# CELL 16: Perplexity evaluation
# ============================================================

@torch.no_grad()
def compute_perplexity(model, data_loader, config):
    """
    Perplexity = exp(average cross-entropy loss)
    Lower = better. 1.0 = perfect (impossible on real data).
    Typical small LM on simple text: 10-50 is decent.
    """
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    pad_id = tokenizer.token_to_id("<pad>")

    for input_ids, targets in data_loader:
        input_ids = input_ids.to(device)
        targets = targets.to(device)

        with autocast(device_type="cuda", enabled=config.fp16):
            logits, _ = model(input_ids)

        # Compute per-token loss (excluding padding)
        loss = torch.nn.functional.cross_entropy(
            logits.view(-1, config.vocab_size),
            targets.view(-1),
            ignore_index=pad_id,
            reduction="sum",
        )

        # Count non-pad tokens
        non_pad = (targets != pad_id).sum().item()
        total_loss += loss.item()
        total_tokens += non_pad

    avg_loss = total_loss / max(total_tokens, 1)
    perplexity = math.exp(avg_loss)

    model.train()
    return perplexity, avg_loss


ppl, avg_loss = compute_perplexity(model, val_loader, config)
print(f"Validation Perplexity: {ppl:.2f}")
print(f"Validation Avg Loss:   {avg_loss:.4f}")

In [ ]:
# ============================================================
# CELL 17: Chat fine-tuning (Phase 2)
# ============================================================
# Convert simple QA pairs into chat format
# Train with loss masked to assistant responses only

class ChatDataset(Dataset):
    """
    Creates chat-formatted training data.
    Loss computed ONLY on assistant tokens.
    """

    def __init__(self, tokenizer, max_len):
        self.max_len = max_len
        self.tokenizer = tokenizer
        self.examples = []
        self.label_masks = []

        # Simple chat examples (in production, load from a dataset)
        chat_pairs = [
            ("What is the sun?", "The sun is a star that gives us light and warmth."),
            ("Tell me a story about a cat.", "Once upon a time, there was a little cat who loved to play in the garden."),
            ("What color is the sky?", "The sky is blue during the day and dark at night."),
            ("Why do birds fly?", "Birds fly because they have wings and light bones that help them stay in the air."),
            ("What is rain?", "Rain is water that falls from clouds in the sky."),
            ("Who is your friend?", "My friend is someone who is kind and likes to play with me."),
            ("What do fish do?", "Fish swim in water. They use their fins to move around."),
            ("Tell me about the moon.", "The moon is bright at night. It goes around the earth."),
            ("What is a tree?", "A tree is a tall plant with leaves, branches, and roots."),
            ("Why do we sleep?", "We sleep so our body can rest and grow strong."),
            ("What is ice?", "Ice is frozen water. It is very cold and hard."),
            ("Where do stars come from?", "Stars are very far away. They are made of hot gas and shine in the night sky."),
            ("What makes a rainbow?", "A rainbow appears when sunlight shines through rain drops in the sky."),
            ("What is a friend?", "A friend is someone who cares about you and likes to spend time with you."),
            ("Why is grass green?", "Grass is green because it has something called chlorophyll that makes it that color."),
        ]

        user_token = "<|user|>"
        asst_token = "<|assistant|>"

        for user_msg, asst_msg in chat_pairs:
            # Format: <|user|>\n{msg}\n<|assistant|>\n{msg}
            full_text = f"{user_token}\n{user_msg}\n{asst_token}\n{asst_msg}"

            # Tokenize full conversation
            full_enc = tokenizer.encode(full_text)
            full_ids = full_enc.ids[:max_len]

            # Tokenize just the user part to find where assistant starts
            user_part = f"{user_token}\n{user_msg}\n{asst_token}\n"
            user_enc = tokenizer.encode(user_part)
            user_len = len(user_enc.ids)

            # Create label mask: -100 for user tokens, real ids for assistant
            labels = []
            for i, token_id in enumerate(full_ids):
                if i < user_len:
                    labels.append(-100)  # mask user tokens
                else:
                    labels.append(token_id)

            # Pad
            pad_id = tokenizer.token_to_id("<pad>")
            while len(full_ids) < max_len:
                full_ids.append(pad_id)
                labels.append(-100)

            self.examples.append(full_ids[:max_len])
            self.label_masks.append(labels[:max_len])

        # Repeat to create more training data
        repeat_factor = 50
        self.examples = self.examples * repeat_factor
        self.label_masks = self.label_masks * repeat_factor

        print(f"Chat dataset: {len(self.examples)} examples")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ids = self.examples[idx]
        labels = self.label_masks[idx]

        x = torch.tensor(ids[:-1], dtype=torch.long)
        y = torch.tensor(labels[1:], dtype=torch.long)
        return x, y


# Build chat dataset
chat_dataset = ChatDataset(tokenizer, config.max_len + 1)
chat_loader = DataLoader(
    chat_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    drop_last=True,
)

# Fine-tune for a few epochs
print("\n--- Phase 2: Chat Fine-Tuning ---")

chat_optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,  # Lower LR for fine-tuning
    betas=(0.9, 0.95),
    weight_decay=0.01,
)

model.train()
chat_epochs = 3

for epoch in range(chat_epochs):
    total_loss = 0
    num_batches = 0

    for input_ids, targets in chat_loader:
        input_ids = input_ids.to(device)
        targets = targets.to(device)

        with autocast(device_type="cuda", enabled=config.fp16):
            logits, _ = model(input_ids)
            loss = torch.nn.functional.cross_entropy(
                logits.view(-1, config.vocab_size),
                targets.view(-1),
                ignore_index=-100,
            )

        chat_optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        chat_optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    avg = total_loss / num_batches
    print(f"  Chat Epoch {epoch+1}/{chat_epochs} | Loss: {avg:.4f}")

print("Chat fine-tuning complete.")

In [ ]:
# ============================================================
# CELL 18: Test chat behavior
# ============================================================

def chat(model, tokenizer, user_message, max_len=150, temperature=0.7):
    """Generate a chat response."""
    prompt = f"<|user|>\n{user_message}\n<|assistant|>\n"
    response = generate_text(
        model, tokenizer, prompt,
        max_len=max_len,
        temperature=temperature,
    )
    # Extract assistant response
    if "<|assistant|>" in response:
        response = response.split("<|assistant|>")[-1].strip()
    return response


print("=" * 60)
print("ZERONET-1: CHAT MODE")
print("=" * 60)

test_questions = [
    "What is the sun?",
    "Tell me a story.",
    "Why is the sky blue?",
    "What do cats like?",
]

for q in test_questions:
    print(f"\nUser: {q}")
    answer = chat(model, tokenizer, q)
    print(f"ZeroNet-1: {answer}")
    print("-" * 40)

In [ ]:
# ============================================================
# CELL 19: Save in Hugging Face format
# ============================================================

import json
import os

def save_for_huggingface(model, tokenizer, config, save_dir):
    """
    Save model in a format compatible with Hugging Face Hub.
    We save the raw PyTorch model + config + tokenizer.
    """
    os.makedirs(save_dir, exist_ok=True)

    # 1. Save model weights
    torch.save(model.state_dict(), os.path.join(save_dir, "pytorch_model.bin"))
    print(f"  Model weights saved.")

    # 2. Save config
    model_config = {
        "architecture": "ZeroNet1",
        "vocab_size": config.vocab_size,
        "d_model": config.d_model,
        "n_heads": config.n_heads,
        "n_layers": config.n_layers,
        "max_len": config.max_len,
        "mlp_ratio": config.mlp_ratio,
        "dropout": config.dropout,
        "rope": True,
        "weight_tying": True,
        "activation": "SwiGLU",
        "norm": "pre-norm LayerNorm",
    }
    with open(os.path.join(save_dir, "config.json"), "w") as f:
        json.dump(model_config, f, indent=2)
    print(f"  Config saved.")

    # 3. Save tokenizer
    tokenizer.save(os.path.join(save_dir, "tokenizer.json"))
    print(f"  Tokenizer saved.")

    # 4. Create model card
    model_card = f"""---
tags:
  - pytorch
  - transformer
  - language-model
  - from-scratch
  - rope
  - decoder-only
license: mit
---

# ZeroNet-1

A small (~{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameter) decoder-only Transformer language model trained from scratch.

## Architecture
- **Type**: Decoder-only causal Transformer
- **Layers**: {config.n_layers}
- **Attention Heads**: {config.n_heads}
- **Hidden Size**: {config.d_model}
- **Context Length**: {config.max_len}
- **Positional Encoding**: RoPE (Rotary Positional Embeddings)
- **Activation**: SwiGLU
- **Norm**: Pre-norm LayerNorm
- **Weight Tying**: Yes (embedding ↔ output head)

## Training
- **Dataset**: TinyStories (from Hugging Face)
- **Tokenizer**: Custom BPE ({config.vocab_size} vocab)
- **Optimizer**: AdamW
- **Precision**: FP16
- **Trained from scratch**: No pretrained weights used

## Usage
```python
# Load and generate
import torch, json
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("tokenizer.json")
# Load model with the ZeroNet1 class definition
model.load_state_dict(torch.load("pytorch_model.bin"))

Limitations
This is a small educational model. It is NOT:

A production system

Capable of factual accuracy

Safe for deployment without filters
"""
with open(os.path.join(save_dir, "README.md"), "w") as f:
  f.write(model_card)
  print(f" Model card saved.")

print(f"\nAll files saved to: {save_dir}/")
for fname in os.listdir(save_dir):
    fsize = os.path.getsize(os.path.join(save_dir, fname))
    print(f" {fname}: {fsize / 1e6:.1f} MB")

In [ ]:
# ============================================================
# CELL 20: Upload to Hugging Face Hub
# ============================================================

from huggingface_hub import HfApi, login

# Login (you need a token from https://huggingface.co/settings/tokens)
login()

def upload_to_hub(local_dir, repo_id):
    """Upload model directory to Hugging Face Hub."""
    api = HfApi()

    # Create repo if it doesn't exist
    api.create_repo(repo_id=repo_id, exist_ok=True, repo_type="model")

    # Upload all files
    api.upload_folder(
        folder_path=local_dir,
        repo_id=repo_id,
        repo_type="model",
    )

    print(f"\n✓ Model uploaded to: https://huggingface.co/{repo_id}")


# CHANGE THIS to your username
repo_id = config.hf_repo  # "YOUR_USERNAME/zeronet-1"

upload_to_hub(config.model_dir, repo_id)

In [ ]:
# ============================================================
# CELL 21: Post-training INT8 quantization
# ============================================================

def quantize_model_int8(model, save_dir):
    """
    Simple post-training dynamic quantization.
    Quantizes Linear layers to INT8.
    ~2-4x smaller, minimal accuracy loss for small models.
    """
    os.makedirs(save_dir, exist_ok=True)

    # Move model to CPU for quantization
    model_cpu = model.cpu()
    model_cpu.eval()

    # Dynamic quantization (no calibration data needed)
    quantized_model = torch.quantization.quantize_dynamic(
        model_cpu,
        {nn.Linear},  # which layers to quantize
        dtype=torch.qint8,
    )

    # Save
    torch.save(quantized_model.state_dict(), os.path.join(save_dir, "pytorch_model_int8.bin"))

    # Compare sizes
    original_size = os.path.getsize(os.path.join(config.model_dir, "pytorch_model.bin"))
    quant_size = os.path.getsize(os.path.join(save_dir, "pytorch_model_int8.bin"))

    print(f"Original model size:  {original_size / 1e6:.1f} MB")
    print(f"Quantized model size: {quant_size / 1e6:.1f} MB")
    print(f"Compression ratio:    {original_size / quant_size:.1f}x")

    # Copy tokenizer and config
    import shutil
    for fname in ["tokenizer.json", "config.json"]:
        src = os.path.join(config.model_dir, fname)
        if os.path.exists(src):
            shutil.copy2(src, save_dir)

    # Move model back to GPU
    model.to(device)

    return quantized_model


quant_dir = config.model_dir + "-int8"
quantized = quantize_model_int8(model, quant_dir)
print(f"\nQuantized model saved to: {quant_dir}/")

In [ ]:
# ============================================================
# CELL 22: Final verification — everything works
# ============================================================

print("=" * 60)
print("ZERONET-1: BUILD COMPLETE")
print("=" * 60)

# Architecture summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"""
MODEL ARCHITECTURE
  Name:           ZeroNet-1
  Type:           Decoder-only causal Transformer
  Parameters:     {total_params:,} ({total_params/1e6:.1f}M)
  Trainable:      {trainable_params:,}
  Vocab size:     {config.vocab_size:,}
  Context length: {config.max_len}
  Layers:         {config.n_layers}
  Attention heads:{config.n_heads}
  Hidden dim:     {config.d_model}
  Head dim:       {config.head_dim}
  MLP ratio:      {config.mlp_ratio}x
  Position enc:   RoPE (Rotary)
  Activation:     SwiGLU
  Normalization:  Pre-norm LayerNorm
  Weight tying:   Yes

TRAINING
  Dataset:        {config.dataset_name}
  Tokenizer:      Custom BPE
  Optimizer:      AdamW
  LR schedule:    Cosine + warmup
  Precision:      {'FP16' if config.fp16 else 'FP32'}
  Epochs:         {config.epochs}

FILES SAVED
""")

for directory in [config.model_dir, config.model_dir + "-int8"]:
    if os.path.exists(directory):
        print(f"  {directory}/")
        for f in sorted(os.listdir(directory)):
            size = os.path.getsize(os.path.join(directory, f))
            print(f"    {f}: {size/1e6:.1f} MB")

# Final generation
print(f"\n{'='*60}")
print("FINAL GENERATION SAMPLES")
print(f"{'='*60}")

final_prompts = [
    "Once upon a time there was",
    "The little bird flew over",
    "She smiled because",
]

for p in final_prompts:
    text = generate_text(model, tokenizer, p, max_len=100, temperature=0.7)
    print(f"\n'{p}' →")
    print(f"  {text}")

print(f"\n{'='*60}")
print("Done. You built an LLM from scratch.")
print(f"{'='*60}")